In [ ]:
!pip install flash-attn --no-build-isolation -q
!pip install pathway sentence-transformers torch -q

In [ ]:
!pip install --upgrade transformers>=4.45.0 -q

In [1]:
# ============================================================================
# NOTEBOOK 2: PATHWAY RETRIEVAL ENGINE
# Using SAME syntax and patterns from Vector Store Builder
# ============================================================================

import pathway as pw
import pandas as pd
import numpy as np
import json
import torch
from sentence_transformers import SentenceTransformer
import gc
import os


2026-01-11 15:14:38.888948: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768144478.903276    1027 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768144478.907378    1027 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768144478.917896    1027 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768144478.917911    1027 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768144478.917913    1027 computation_placer.cc:177] computation placer alr

In [6]:
# ============================================================================
# STEP 1: Configuration
# ============================================================================

EMBEDDING_CONFIG = {
    'model_name': 'Alibaba-NLP/gte-Qwen2-7B-instruct',
    'dimension': 3584,
    'batch_size': 4,
    'trust_remote_code': True,
    'normalize': True,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# ✅ Your corrected paths
DATASET_NAME = '/kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen'
DATASET_BASE = f'{DATASET_NAME}/kaggle/working/pathway_storage'
METADATA_PATH = f"{DATASET_BASE}/vector_store_metadata.json"

device = EMBEDDING_CONFIG['device']

print(f"✅ Device: {device}")
print(f"✅ Model: {EMBEDDING_CONFIG['model_name']}")
print(f"✅ Dataset: {DATASET_NAME}")
print(f"✅ Base: {DATASET_BASE}")


✅ Device: cuda
✅ Model: Alibaba-NLP/gte-Qwen2-7B-instruct
✅ Dataset: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen
✅ Base: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage


In [7]:
# ============================================================================
# STEP 2: Load Metadata
# ============================================================================

print(f"\n📁 Loading metadata from: {METADATA_PATH}")

with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"✅ Metadata loaded")
print(f"   Books: {list(metadata['books'].keys())}")


📁 Loading metadata from: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage/vector_store_metadata.json
✅ Metadata loaded
   Books: ['In-search-of-the-castaways', 'The-count-of-monte-cristo']


In [8]:
# ============================================================================
# STEP 3: FIX PATHS (Debug Version)
# ============================================================================

print("\n🔧 Fixing paths in metadata...")
print(f"   Target base: {DATASET_BASE}")

for book_name in metadata['books'].keys():
    print(f"\n   📖 {book_name}:")
    
    # BEFORE
    old_emb = metadata['books'][book_name]['embeddings_npy']
    old_chunks = metadata['books'][book_name]['chunks_csv']
    
    print(f"      BEFORE:")
    print(f"        Embeddings: {old_emb}")
    print(f"        Chunks: {old_chunks}")
    
    # REPLACE
    new_emb = f"{DATASET_BASE}/embeddings_{book_name}.npy"
    new_chunks = f"{DATASET_BASE}/chunks_{book_name}.csv"
    
    metadata['books'][book_name]['embeddings_npy'] = new_emb
    metadata['books'][book_name]['chunks_csv'] = new_chunks
    
    print(f"      AFTER:")
    print(f"        Embeddings: {new_emb}")
    print(f"        Chunks: {new_chunks}")
    
    # VERIFY
    emb_exists = os.path.exists(new_emb)
    chunks_exists = os.path.exists(new_chunks)
    
    print(f"      VERIFY:")
    print(f"        Embeddings exists: {'✅' if emb_exists else '❌'}")
    print(f"        Chunks exists: {'✅' if chunks_exists else '❌'}")
    
    if not emb_exists:
        print(f"        ❌ ERROR: File not found: {new_emb}")
    if not chunks_exists:
        print(f"        ❌ ERROR: File not found: {new_chunks}")

# Fix parquet paths
if 'pathway_tables' in metadata:
    for book_name in metadata['pathway_tables'].keys():
        old_parquet = metadata['pathway_tables'][book_name]['parquet_path']
        new_parquet = f"{DATASET_BASE}/pathway_chunks_{book_name}.parquet"
        metadata['pathway_tables'][book_name]['parquet_path'] = new_parquet

print("\n✅ Path replacement complete!")



🔧 Fixing paths in metadata...
   Target base: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage

   📖 In-search-of-the-castaways:
      BEFORE:
        Embeddings: /kaggle/working/pathway_storage/embeddings_In-search-of-the-castaways.npy
        Chunks: /kaggle/working/pathway_storage/chunks_In-search-of-the-castaways.csv
      AFTER:
        Embeddings: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage/embeddings_In-search-of-the-castaways.npy
        Chunks: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage/chunks_In-search-of-the-castaways.csv
      VERIFY:
        Embeddings exists: ✅
        Chunks exists: ✅

   📖 The-count-of-monte-cristo:
      BEFORE:
        Embeddings: /kaggle/working/pathway_storage/embeddings_The-count-of-monte-cristo.npy
        Chunks: /kaggle/working/pathway_storage/chunks_The-count-of

In [9]:
# ============================================================================
# STEP 4: Schemas
# ============================================================================

class ChunkWithEmbeddingSchema(pw.Schema):
    chunk_id: int
    book: str
    chunk: str
    chunk_length: int
    embedding: list
    embedding_dim: int
    score: float
    created_at: str

In [10]:
# ============================================================================
# STEP 5: Retrieval Engine
# ============================================================================

class PathwayRetrievalEngine:
    def __init__(self, metadata):
        self.metadata = metadata
        self.device = device
        self.embeddings = {}
        self.chunks_df = {}
        
        print("\n📚 Loading vector stores...")
        self._load_stores()
        
        print(f"\n📦 Loading {EMBEDDING_CONFIG['model_name']}...")
        
        # THE FIX: Move use_cache into config_kwargs
        # This prevents the TypeError and still fixes the DynamicCache error
        self.embedding_model = SentenceTransformer(
            EMBEDDING_CONFIG['model_name'],
            device=device,
            trust_remote_code=True,
            config_kwargs={"use_cache": False} # <--- Changed from model_kwargs
        )
        
        print(f"✅ Ready!")

    # ... _load_stores stays the same ...

    def _get_query_embedding(self, query):
        """Embed query on-the-fly."""
        
        return self.embedding_model.encode(
            [query],
            batch_size=1,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
            # FIX 2: Removed model_kwargs from here to prevent ValueError
        )[0]
        
    def _load_stores(self):
        """Load stores."""
        
        for book_name, book_meta in self.metadata['books'].items():
            emb_path = book_meta['embeddings_npy']
            chunks_path = book_meta['chunks_csv']
            
            print(f"\n   Loading {book_name}...")
            print(f"     Path: {emb_path}")
            
            # Load
            self.embeddings[book_name] = np.load(emb_path)
            self.chunks_df[book_name] = pd.read_csv(chunks_path)
            
            print(f"     ✅ Shape: {self.embeddings[book_name].shape}")
    

    
    def search_streaming(self, query, book_name=None, top_k=3):
        """Search."""
        
        query_embedding = self._get_query_embedding(query)
        
        results = []
        books = [book_name] if book_name else self.embeddings.keys()
        
        for book in books:
            book_embs = self.embeddings[book]
            book_chunks = self.chunks_df[book]
            
            similarities = np.dot(book_embs, query_embedding)
            top_indices = np.argsort(similarities)[-top_k:][::-1]
            
            for rank, idx in enumerate(top_indices):
                results.append({
                    'chunk_id': int(book_chunks.iloc[idx]['chunk_id']),
                    'book': book,
                    'chunk': book_chunks.iloc[idx]['chunk'],
                    'score': float(similarities[idx]),
                    'rank': rank + 1
                })
        
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:top_k]


In [11]:
# ============================================================================
# STEP 6: Initialize
# ============================================================================

print("\n" + "=" * 100)
print("🚀 RETRIEVAL ENGINE")
print("=" * 100)

retriever = PathwayRetrievalEngine(metadata)

# Test
print("\n🔍 Test:")
results = retriever.search_streaming("Edmond Dantes imprisoned", top_k=2)

for i, r in enumerate(results):
    print(f"\n{i+1}. {r['book']} (Score: {r['score']:.4f})")
    print(f"   {r['chunk'][:150]}...")

print("\n✅ Ready for Notebook 3!")


🚀 RETRIEVAL ENGINE

📚 Loading vector stores...

   Loading In-search-of-the-castaways...
     Path: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage/embeddings_In-search-of-the-castaways.npy
     ✅ Shape: (375, 3584)

   Loading The-count-of-monte-cristo...
     Path: /kaggle/input/d/priyanshkeshari/kdsh26-pathway-vector-embeddings-books-qwen/kaggle/working/pathway_storage/embeddings_The-count-of-monte-cristo.npy
     ✅ Shape: (1290, 3584)

📦 Loading Alibaba-NLP/gte-Qwen2-7B-instruct...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Ready!

🔍 Test:

1. The-count-of-monte-cristo (Score: 0.6059)
   to his neighbor. All day he toiled on untiringly, and by the evening he had succeeded in extracting ten handfuls of plaster and fragments of stone. Wh...

2. The-count-of-monte-cristo (Score: 0.5969)
   you are the person I am in search of.” “What proofs do you require?” “Did you, in the year 1814 or 1815, know anything of a young sailor named Dantès?...

✅ Ready for Notebook 3!


In [12]:
# ============================================================================
# STEP 7: Test Searches
# ============================================================================

print("\n" + "=" * 100)
print("📊 TEST SEARCHES")
print("=" * 100)

test_queries = [
    "Edmond Dantes is imprisoned in Chateau d'If",
    "Captain Grant searches for his father in the ocean",
    "The treasure is hidden on Monte Cristo island"
]

print("\nRunning test searches with on-the-fly query embeddings...\n")

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    results = retriever.search_streaming(query, top_k=2)
    
    if results:
        for i, r in enumerate(results):
            print(f"\n   📌 Result {i+1}:")
            print(f"      Book: {r['book']}")
            print(f"      Score: {r['score']:.4f}")
            print(f"      Relevance: {'🔥 High' if r['score'] > 0.5 else '📊 Medium'}")
            print(f"      Text: {r['chunk'][:150]}...")



📊 TEST SEARCHES

Running test searches with on-the-fly query embeddings...


🔍 Query: 'Edmond Dantes is imprisoned in Chateau d'If'

   📌 Result 1:
      Book: The-count-of-monte-cristo
      Score: 0.6015
      Relevance: 🔥 High
      Text: with the wind. In spite of his repugnance to address the guards, Dantès turned to the nearest gendarme, and taking his hand, “Comrade,” said he, “I ad...

   📌 Result 2:
      Book: The-count-of-monte-cristo
      Score: 0.5581
      Relevance: 🔥 High
      Text: “Of what country?” “A Frenchman.” “Your name?” “Edmond Dantès.” “Your profession?” “A sailor.” “How long have you been here?” “Since the 28th of Febru...

🔍 Query: 'Captain Grant searches for his father in the ocean'

   📌 Result 1:
      Book: In-search-of-the-castaways
      Score: 0.4521
      Relevance: 📊 Medium
      Text: earnest and skilled oarsmen sped away towards the shore. At ten yards therefrom, Mary uttered again the heart-rending cry: "My father!" A man was stan...

   📌 Res

In [13]:
# ============================================================================
# STEP 8: Summary
# ============================================================================

print("\n" + "=" * 100)
print("✅ RETRIEVAL ENGINE READY")
print("=" * 100)

print(f"\n📚 Loaded vector stores:")
print(f"   Model: {EMBEDDING_CONFIG['model_name']}")
print(f"   Dimension: 3584 (actual)")
print(f"   Books: {list(metadata['books'].keys())}")
print(f"   Total chunks: {sum(meta['num_chunks'] for meta in metadata['books'].values())}")

print(f"\n🎯 How it works:")
print(f"   1. Book embeddings: Pre-computed ✅ (stored in dataset)")
print(f"   2. Query embeddings: On-the-fly ✅ (computed when searching)")
print(f"   3. Similarity: Cosine between query and all book chunks")
print(f"   4. Results: Top-k most relevant passages")

print(f"\n✅ Ready for Notebook 3 (Rationale Generator)!")


✅ RETRIEVAL ENGINE READY

📚 Loaded vector stores:
   Model: Alibaba-NLP/gte-Qwen2-7B-instruct
   Dimension: 3584 (actual)
   Books: ['In-search-of-the-castaways', 'The-count-of-monte-cristo']
   Total chunks: 1665

🎯 How it works:
   1. Book embeddings: Pre-computed ✅ (stored in dataset)
   2. Query embeddings: On-the-fly ✅ (computed when searching)
   3. Similarity: Cosine between query and all book chunks
   4. Results: Top-k most relevant passages

✅ Ready for Notebook 3 (Rationale Generator)!
